# ЛР 2. Заполнение пропусков и кодирование категориальных признаков

**Вариант:** Marvel, учебный файл `marvel.csv`. Это продолжение [ЛР 1](../LR1_marvel/LR1_marvel.ipynb). Используются сохранённые там обучающая и тестовая части; новое разделение не выполняется.

**Цель:** сравнить статистические и машинные методы импутации, выбрать способ для каждого неполного признака, затем закодировать номинальные категории. Требования: [страница ЛР 2 ПМиФИ](https://wiki.pmifi.ru/ru/disciplines/programming-workshop-bachelors-5sem/labs/imputation-cat).

## Содержание

1. [Исходные данные и границы этапа](#data)
2. [Сравнение методов заполнения](#comparison)
3. [Заполнение обучающей и тестовой частей](#imputation)
4. [Кодирование категорий](#encoding)
5. [Проверка и сохранение результата](#result)

## 1. Исходные данные и границы этапа <a id="data"></a>

ЛР 1 создала `splits/X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`. Столбец `source_row` в этих файлах — исходный индекс строки, а не признак. Мы сохраняем разбиение **614 / 154** и два целевых столбца без изменений.

Пропуски есть в `Relative Compactness`, `Reactor Wall Area`, `Overall Height`, `Orientation`. Первые три признака рассматриваются как числовые, а `Orientation` — как номинальный код. `Glazing Area Distribution` тоже номинальный, хотя пропусков в нём нет. Значения целей не используются для восстановления входов: в момент будущего прогноза цели неизвестны.

**Защита от утечки:** сравнение подходов и настройка преобразований выполняются только на `X_train`. К `X_test` применяются параметры, полученные на обучающей части. Итоговая оценка качества модели остаётся для следующих работ.

In [1]:
from pathlib import Path
import platform
from importlib.metadata import version

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
SOURCE_DIR = Path("../LR1_marvel/splits")
if not SOURCE_DIR.exists():
    SOURCE_DIR = Path("LR1_marvel/splits")
assert SOURCE_DIR.exists(), "Не найдены результаты ЛР 1. Запустите ноутбук из папки ЛР 2 или корня репозитория."

print("Python:", platform.python_version())
for package in ["numpy", "pandas", "scikit-learn"]:
    print(package, version(package))
print("Источник:", SOURCE_DIR.resolve())

Python: 3.12.5
numpy 2.4.5
pandas 3.0.2
scikit-learn 1.8.0
Источник: /Users/arsenijlogunov/Documents/ChatGPT/Практикум по програмированию/LR1_marvel/splits


In [2]:
X_train = pd.read_csv(SOURCE_DIR / "X_train.csv", sep=";", index_col="source_row")
X_test = pd.read_csv(SOURCE_DIR / "X_test.csv", sep=";", index_col="source_row")
y_train = pd.read_csv(SOURCE_DIR / "y_train.csv", sep=";", index_col="source_row")
y_test = pd.read_csv(SOURCE_DIR / "y_test.csv", sep=";", index_col="source_row")

categorical_columns = ["Orientation", "Glazing Area Distribution"]
missing_columns = ["Relative Compactness", "Reactor Wall Area", "Overall Height", "Orientation"]
numeric_columns = [column for column in X_train if column not in categorical_columns]

assert X_train.shape == (614, 8) and X_test.shape == (154, 8)
assert y_train.shape == (614, 2) and y_test.shape == (154, 2)
assert set(X_train.index).isdisjoint(X_test.index)
assert X_train.columns.equals(X_test.columns)
assert y_train.notna().all().all() and y_test.notna().all().all()

print("Входы:", X_train.shape, X_test.shape)
display(pd.DataFrame({"train: пропуски": X_train.isna().sum(), "test: пропуски": X_test.isna().sum()}))

Входы: (614, 8) (154, 8)


,train: пропуски,test: пропуски
Relative Compactness,64,13
Surface Area,0,0
Reactor Wall Area,65,12
Reactor Roof Area,0,0
Overall Height,63,14
Orientation,61,16
Glazing Area,0,0
Glazing Area Distribution,0,0


## 2. Сравнение методов заполнения <a id="comparison"></a>

Проверить качество импутации на настоящих пропусках нельзя: их правильные значения неизвестны. Поэтому для каждого столбца мы **искусственно скрываем 20% известных значений** в обучающей части, восстанавливаем их и сравниваем с исходными. Повторяем три раза с фиксированными `random_state` 42, 43, 44. Настоящий тест в подборе не участвует.

| Метод | Класс | Принцип |
|---|---|---|
| Медиана для чисел / мода для `Orientation` | Статистический | Подставляет одно типичное значение из обучающей части |
| KNN, 5 соседей | Машинный | Ищет похожие строки по стандартизованным признакам и усредняет известные значения |
| Случайный лес | Машинный | Учит зависимость восстанавливаемого признака от остальных входов; для `Orientation` используется классификатор |

Для числовых столбцов ошибка — **MAE** (меньше лучше); для `Orientation` — **accuracy** по кодам (больше лучше). В лесу пропуски остальных предикторов сначала заполняются медианой или модой, вычисленной на строках, доступных для обучения данного леса. KNN трактует числовые коды категорий как координаты при поиске соседей — это его ограничение, которое учитывается при интерпретации.

In [3]:
def predictor_fill_values(frame, columns, categorical):
    return {
        column: (frame[column].mode(dropna=True).iloc[0] if column in categorical
                 else frame[column].median())
        for column in columns
    }


def predict_missing(train_frame, prediction_frame, column, method, categorical):
    prediction_rows = prediction_frame[column].isna()
    if not prediction_rows.any():
        return pd.Series(dtype=float, index=prediction_frame.index[prediction_rows])

    if method == "Медиана / мода":
        strategy = "most_frequent" if column in categorical else "median"
        imputer = SimpleImputer(strategy=strategy).fit(train_frame[[column]])
        values = imputer.transform(prediction_frame[[column]])[:, 0]
        return pd.Series(values, index=prediction_frame.index).loc[prediction_rows]

    if method == "KNN":
        scaler = StandardScaler().fit(train_frame)
        scaled_train = scaler.transform(train_frame)
        scaled_prediction = scaler.transform(prediction_frame)
        imputer = KNNImputer(n_neighbors=5, weights="distance").fit(scaled_train)
        restored = scaler.inverse_transform(imputer.transform(scaled_prediction))
        values = pd.Series(restored[:, train_frame.columns.get_loc(column)], index=prediction_frame.index)
        if column in categorical:
            levels = np.sort(train_frame[column].dropna().unique())
            values = pd.Series(levels[np.abs(values.to_numpy()[:, None] - levels).argmin(axis=1)], index=values.index)
        return values.loc[prediction_rows]

    predictors = [name for name in train_frame if name != column]
    known = train_frame[column].notna()
    fill_values = predictor_fill_values(train_frame.loc[known], predictors, categorical)
    fit_X = train_frame.loc[known, predictors].fillna(fill_values)
    predict_X = prediction_frame.loc[prediction_rows, predictors].fillna(fill_values)
    model_class = RandomForestClassifier if column in categorical else RandomForestRegressor
    model = model_class(n_estimators=100, min_samples_leaf=2, random_state=SEED, n_jobs=-1)
    model.fit(fit_X, train_frame.loc[known, column])
    return pd.Series(model.predict(predict_X), index=predict_X.index)

In [4]:
records = []
for column in missing_columns:
    observed_rows = X_train.index[X_train[column].notna()]
    for repeat in range(3):
        _, masked_rows = train_test_split(observed_rows, test_size=0.20, random_state=SEED + repeat)
        masked_frame = X_train.copy()
        masked_frame.loc[masked_rows, column] = np.nan
        truth = X_train.loc[masked_rows, column]
        for method in ["Медиана / мода", "KNN", "Случайный лес"]:
            prediction = predict_missing(masked_frame, masked_frame, column, method, categorical_columns).loc[masked_rows]
            score = (accuracy_score(truth, prediction) if column in categorical_columns
                     else mean_absolute_error(truth, prediction))
            records.append({"Признак": column, "Повтор": repeat + 1, "Метод": method, "Метрика": "accuracy" if column in categorical_columns else "MAE", "Значение": score})

comparison = pd.DataFrame(records)
summary = comparison.groupby(["Признак", "Метод", "Метрика"], as_index=False)["Значение"].mean()
summary = summary.rename(columns={"Значение": "Среднее за 3 повтора"})
display(summary.round(4))

,Признак,Метод,Метрика,Среднее за 3 повтора
0,Orientation,KNN,accuracy,0.1111
1,Orientation,Медиана / мода,accuracy,0.2192
2,Orientation,Случайный лес,accuracy,0.0571
3,Overall Height,KNN,MAE,0.0000
4,Overall Height,Медиана / мода,MAE,1.8814
5,Overall Height,Случайный лес,MAE,0.0000
6,Reactor Wall Area,KNN,MAE,13.1210
7,Reactor Wall Area,Медиана / мода,MAE,33.5576
8,Reactor Wall Area,Случайный лес,MAE,0.1654
9,Relative Compactness,KNN,MAE,0.0098


In [5]:
selected_methods = {}
for column in missing_columns:
    part = summary.loc[summary["Признак"] == column]
    best_row = (part["Среднее за 3 повтора"].idxmax() if column in categorical_columns
                else part["Среднее за 3 повтора"].idxmin())
    selected_methods[column] = summary.loc[best_row, "Метод"]

selection = pd.DataFrame({"Признак": list(selected_methods), "Выбранный метод": list(selected_methods.values())})
display(selection)

,Признак,Выбранный метод
0,Relative Compactness,Случайный лес
1,Reactor Wall Area,Случайный лес
2,Overall Height,Случайный лес
3,Orientation,Медиана / мода


**Выбор по результатам трёх повторов (только train):**

- `Relative Compactness`: случайный лес, MAE **0,00008** против **0,00979** у KNN и **0,08267** у медианы.
- `Reactor Wall Area`: случайный лес, MAE **0,165** против **13,121** у KNN и **33,558** у медианы.
- `Overall Height`: случайный лес и KNN практически без ошибки; выбран лес как единый метод для геометрических столбцов. Медиана даёт MAE **1,881**.
- `Orientation`: мода, accuracy **0,219** против **0,111** у KNN и **0,057** у леса. Даже лучший из этих методов угадывает ориентацию слабо: по остальным характеристикам она почти не восстанавливается. Значения, подставленные вместо пропусков, следует трактовать как приближение.

MAE разных столбцов нельзя сравнивать напрямую между собой, так как у них разные масштабы. Сравнивается один и тот же столбец между методами. Это оценка восстановления искусственно скрытых известных значений, а не качества будущей модели прогноза нагрузок.

## 3. Заполнение обучающей и тестовой частей <a id="imputation"></a>

Для каждого столбца выбранный метод заново настраивается на **полной обучающей части**. При вычислении подстановок для теста используются только параметры и модели из train. Восстанавливаются лишь отсутствующие значения; наблюдавшиеся значения сохраняются. Целевые `y_train` и `y_test` остаются исходными.

In [6]:
X_train_filled = X_train.copy()
X_test_filled = X_test.copy()
for column, method in selected_methods.items():
    X_train_filled.loc[X_train[column].isna(), column] = predict_missing(
        X_train, X_train, column, method, categorical_columns
    )
    X_test_filled.loc[X_test[column].isna(), column] = predict_missing(
        X_train, X_test, column, method, categorical_columns
    )

assert not X_train_filled.isna().any().any()
assert not X_test_filled.isna().any().any()
for column in X_train.columns:
    assert X_train_filled.loc[X_train[column].notna(), column].equals(X_train.loc[X_train[column].notna(), column])
    assert X_test_filled.loc[X_test[column].notna(), column].equals(X_test.loc[X_test[column].notna(), column])

print("Пропуски после заполнения:", int(X_train_filled.isna().sum().sum()), int(X_test_filled.isna().sum().sum()))
display(X_train_filled[missing_columns].describe().T.round(3))

Пропуски после заполнения: 0 0


,count,mean,std,min,25%,50%,75%,max
Relative Compactness,614.0,0.762,0.105,0.62,0.66,0.74,0.82,0.98
Reactor Wall Area,614.0,318.819,43.336,245.00,294.00,318.50,343.00,416.50
Overall Height,614.0,5.221,1.751,3.50,3.50,3.50,7.00,7.00
Orientation,614.0,3.664,1.149,2.00,3.00,4.00,5.00,5.00


## 4. Кодирование категорий <a id="encoding"></a>

`Orientation` и `Glazing Area Distribution` — **номинальные** признаки: коды обозначают варианты, но не порядок или расстояние. Поэтому выбираем **one-hot encoding**: каждому наблюдаемому коду соответствует отдельный бинарный столбец. Простое оставление кодов 2–5 и 0–5 создало бы для многих моделей искусственный порядок. `Glazing Area` остаётся числовой долей, хотя в этом файле имеет лишь несколько уровней.

Кодировщик обучается на train. `handle_unknown="ignore"` позволяет обработать ранее не встреченный код в test без падения программы; он получит нули во всех столбцах своей категории. `drop=None` сохраняет все уровни. Для некоторых линейных моделей позже может понадобиться удалить один уровень, но выбор модели относится к следующему этапу.

In [7]:
category_train = X_train_filled[categorical_columns].astype("Int64").astype(str)
category_test = X_test_filled[categorical_columns].astype("Int64").astype(str)
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.int8)
encoded_train = encoder.fit_transform(category_train)
encoded_test = encoder.transform(category_test)
encoded_columns = encoder.get_feature_names_out(categorical_columns).tolist()

X_train_ready = pd.concat([
    X_train_filled[numeric_columns],
    pd.DataFrame(encoded_train, columns=encoded_columns, index=X_train_filled.index)
], axis=1)
X_test_ready = pd.concat([
    X_test_filled[numeric_columns],
    pd.DataFrame(encoded_test, columns=encoded_columns, index=X_test_filled.index)
], axis=1)

print("Категории, найденные на train:")
for column, levels in zip(categorical_columns, encoder.categories_):
    print(column, levels.tolist())
print("Размер после кодирования:", X_train_ready.shape, X_test_ready.shape)
display(X_train_ready.head())

Категории, найденные на train:
Orientation ['2', '3', '4', '5']
Glazing Area Distribution ['0', '1', '2', '3', '4', '5']
Размер после кодирования: (614, 16) (154, 16)


,Relative Compactness,Surface Area,Reactor Wall Area,Reactor Roof Area,Overall Height,Glazing Area,Orientation_2,Orientation_3,Orientation_4,Orientation_5,Glazing Area Distribution_0,Glazing Area Distribution_1,Glazing Area Distribution_2,Glazing Area Distribution_3,Glazing Area Distribution_4,Glazing Area Distribution_5
source_row,,,,,,,,,,,,,,,,
60,0.82,612.5,318.5,147.0,7.0,0.10,1,0,0,0,0,1,0,0,0,0
618,0.64,784.0,343.0,220.5,3.5,0.40,0,0,1,0,0,0,1,0,0,0
346,0.86,588.0,294.0,147.0,7.0,0.25,0,0,0,1,0,0,1,0,0,0
294,0.90,563.5,318.5,122.5,7.0,0.25,0,0,1,0,0,1,0,0,0,0
231,0.66,759.5,318.5,220.5,3.5,0.10,0,0,0,1,0,0,0,0,1,0


## 5. Проверка и сохранение результата <a id="result"></a>

Проверяем одинаковый состав столбцов, отсутствие пропусков, числовые типы и непересечение исходных индексов. Сохраняем подготовленные входы и неизменённые цели в `processed/`. `source_row` в CSV нужен для сопоставления строк и не должен становиться входом модели. Результаты сравнения методов также сохраняются для воспроизводимости решения.

In [8]:
assert X_train_ready.shape[0] == y_train.shape[0] == 614
assert X_test_ready.shape[0] == y_test.shape[0] == 154
assert X_train_ready.columns.equals(X_test_ready.columns)
assert not X_train_ready.isna().any().any() and not X_test_ready.isna().any().any()
assert all(pd.api.types.is_numeric_dtype(dtype) for dtype in X_train_ready.dtypes)
assert set(X_train_ready.index).isdisjoint(X_test_ready.index)
assert y_train.equals(pd.read_csv(SOURCE_DIR / "y_train.csv", sep=";", index_col="source_row"))
assert y_test.equals(pd.read_csv(SOURCE_DIR / "y_test.csv", sep=";", index_col="source_row"))

OUTPUT_DIR = Path("processed")
OUTPUT_DIR.mkdir(exist_ok=True)
for name, frame in {"X_train": X_train_ready, "X_test": X_test_ready, "y_train": y_train, "y_test": y_test}.items():
    frame.to_csv(OUTPUT_DIR / f"{name}.csv", sep=";", index_label="source_row")
summary.to_csv(OUTPUT_DIR / "imputation_comparison.csv", sep=";", index=False)
selection.to_csv(OUTPUT_DIR / "selected_methods.csv", sep=";", index=False)

print("Подготовленные входы:", X_train_ready.shape, X_test_ready.shape)
print("Пропусков:", int(X_train_ready.isna().sum().sum()), int(X_test_ready.isna().sum().sum()))
print("Сохранено в:", OUTPUT_DIR.resolve())

Подготовленные входы: (614, 16) (154, 16)
Пропусков: 0 0
Сохранено в: /Users/arsenijlogunov/Documents/ChatGPT/Практикум по програмированию/LR2_marvel/processed


### Итог

- Статистическая импутация сравнена с KNN и случайным лесом на искусственно скрытых **известных значениях только обучающей части**.
- Для каждого неполного признака выбран метод по MAE либо accuracy, выбор записан в таблице выше.
- Пропуски в train и test устранены, категории закодированы one-hot, цели не изменены.
- Эта работа подготавливает данные к следующим этапам. Она **не измеряет** качество модели прогноза отопительной и охлаждающей нагрузки. Тестовая часть остаётся для итоговой проверки будущей модели.